# 04 — Design and interpret carbon and alkalinity experiments

**Learning goals:** interpret how carbon and alkalinity inputs change atm CO2, ocean chemistry and carbonate preservation; explain critical-depth motion and OA/OAE asymmetry using matched controls.

**Provisional time: 40 minutes.** Prediction/specification (5), two forcing tasks (10), supplied runs/checks (10), four short answers (15). Three cases are run: control, OA and OAE. All plotting and numerical machinery are supplied.
[Teaching goals](../../TEACHING_GOALS.md).

**Reading key:** <mark style="background-color: #fff0b3; color: #513d00; padding: 0.05em 0.2em; border-radius: 3px;"><strong>Key term</strong></mark> = concept to notice. Blue **Question** panels identify student work; purple **Instructor answer** panels appear only in the instructor sheet.
Code labels distinguish **Choose and explain**, **Understand and run**, and **Supplied implementation**.
This practical is ungraded. Keep your explanations here; no separate submission is required.
The [coding cheatsheet](../../ref/modelling_cheatsheet.md) is optional lookup support; essential syntax is explained locally.

In [ ]:
# Supplied implementation: run this support code as provided.
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'teaching_config.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from esbmtk import Signal, Source, Species2Species
from model import initialize_model, postprocess_carbonate_horizons, run_model
from presets import load_boudreau_parameters, make_pump_variant
from model_inputs import read_model_tables

DATA = ROOT / 'data' / 'Boudreau_2010'
WORKBOOK = DATA / 'model_definition.xlsx'
P = load_boudreau_parameters(WORKBOOK)
input_tables = read_model_tables(WORKBOOK)
STATE = DATA / 'steady_state'
PULSE_FILE = DATA / 'IS92a-scenario.csv'
DIGITIZED = DATA / 'digitized'
REFERENCE_SCALE = 0.877
PULSE_START = 1800.0
REFERENCE_CARBON_PMOL = 335.3560189847107
from teaching_audits import audit_complete_model

## A1. Reuse the model specification from 03

Use your completed 03 schematic as the experiment map; add the external input arrow. Each case uses an independent copy of the same workbook geometry, box-specific T/S/P, transport and baseline rates, and the same saved nearly stationary state (the **restart**). These saved concentrations replace the workbook initial values.

POC/PIC export and weathering remain fixed; gas exchange and carbonate dissolution/burial respond to the evolving state. The atm reservoir is finite. **Ocean acidification (OA)** here is driven by carbon input; idealized **ocean alkalinity enhancement (OAE)** supplies TA with no direct carbon input.

OA uses the archived IS92a-shaped forcing (scale 0.877, no terrestrial uptake). OAE uses the same time profile, with its own prescribed amount. These are a historical benchmark scenario and a teaching intervention, not present-day forecasts. Runs cover model years 0–3800 with a one-month maximum step; year 1800 is the reporting reference, not a switch from exactly zero input.

After changing a forcing choice, rerun the case-building cell and all subsequent cells. Changing baseline geometry, chemistry or process rates requires a new compatible stationary restart.

<div style="background-color: #edf5ff; color: #173b61; border-left: 4px solid #3977b8; padding: 12px 16px; margin: 16px 0; border-radius: 4px;">

**Question — specify and predict before running**

Complete the entry-species/receiving-state fields through Exercise 04.2 below and your diagram. State what must match the control. Predict the sign of atm CO2 and surface pH anomalies in OA and OAE; identify a diagnostic whose opposite sign would challenge your prediction. Revisit this prediction in C1.

| Case | External input after year 1800 | Entry species / receiving state | Initial state and baseline |
| --- | --- | --- | --- |
| Control | None | No forcing connection | Archived restart; fixed benchmark parameters |
| OA | About 4025 Gt C | Your diagram and code choice | Same as control |
| OAE | 10 Pmol TA equivalents | Your diagram and code choice | Same as control |

The whole-run forcing includes a small pre-1800 tail. The supplied audit reports both intervals. The amounts and units differ, so raw OA/OAE curves do not compare equal-strength interventions.

</div>

> **Your explanation:** replace this placeholder with your answer.

<details>
<summary>Optional reference — inspect the shared workbook again</summary>

The following function displays input records only if you call it. The core uses the specification you already inspected in 03. Numerical baseline inputs stay in `model_definition.xlsx`.

</details>

In [ ]:
# Supplied implementation: optional lookup, not another required table-reading task.
def show_input_reference():
    display(pd.DataFrame(input_tables['OceanReservoirs']).set_index('Box ID'))
    display(pd.DataFrame(input_tables['Atmosphere']).set_index('Box ID'))
    display(pd.DataFrame(input_tables['TransportConnections']).sort_values('Order'))
    display(pd.DataFrame(input_tables['GasExchangeConnections']).sort_values('Order'))
    display(pd.DataFrame(input_tables['ProcessParameters']).set_index('Parameter'))
# To inspect the tables again, run show_input_reference().

## A2. Exercise 04.1: specify the two forcing inventories

Use the amounts in A1's table for the interval after model year 1800.

<div style="background-color: #edf5ff; color: #173b61; border-left: 4px solid #3977b8; padding: 12px 16px; margin: 16px 0; border-radius: 4px;">

**Question — specify forcing inventories**

Assign `oa_target_mol` and `oae_target_mol` in mol C and mol TA equivalents. One Gt is $10^{15}$ g, one Pmol is $10^{15}$ mol, and carbon has molar mass 12 g/mol. The supplied scaling preserves the archived shape and its small pre-1800 tail; B1 reports both the post-1800 and whole-run inputs.

</div>

In [ ]:
# Choose and explain: complete marked choices; surrounding machinery is supplied.
# Exercise 04.1: convert the prescribed amounts to the model's mol units.
raise NotImplementedError("Exercise: replace this line with your solution")
# Supplied conversion from inventory to the archived signal's scale factor.
# Keep its exact reference normalization; 4025 Gt is a rounded comparison target.
np.testing.assert_allclose(oa_target_mol / 1e15, REFERENCE_CARBON_PMOL, rtol=1e-3)
oae_scale = REFERENCE_SCALE * oae_target_mol / (REFERENCE_CARBON_PMOL * 1e15)

## A3. Exercise 04.2: map the external arrows

<div style="background-color: #edf5ff; color: #173b61; border-left: 4px solid #3977b8; padding: 12px 16px; margin: 16px 0; border-radius: 4px;">

**Question — implement the experiment specification**

Inside each branch, assign `forcing_species` and `forcing_target` from your diagram. Choose from the model's CO2 and TA species and its atmospheric CO2 or low-latitude surface TA state. No pH is imposed; it is calculated from the evolving state.

`Source` represents material outside the boundary. `Signal` supplies a time-dependent flux; the native connection attaches it to the chosen state. The control has no such connection. Constructor syntax, waveform scaling and the numerical audit are supplied.

</div>

In [ ]:
# Choose and explain: complete marked choices; surrounding machinery is supplied.
def build_complete_case(forcing=None):
    # Supplied model and stationary restart; pumps remain fixed.
    params = make_pump_variant(base=P)  # independent copy with fixed baseline export
    model = initialize_model(params, stop='3800 yr', max_timestep='1 month')
    model.read_state(directory=str(STATE))
    if forcing is None:
        return model
    # Exercise 04.2: set the species and receiving state in both cases.
    if forcing == 'OA':
        raise NotImplementedError("Exercise: replace this line with your solution")
    elif forcing == 'OAE':
        raise NotImplementedError("Exercise: replace this line with your solution")
    else:
        raise ValueError(forcing)
    scale = {'OA': REFERENCE_SCALE, 'OAE': oae_scale}[forcing]
    signal = Signal(name='external_input', species=forcing_species, register=model,
                    filename=str(PULSE_FILE), scale=scale)
    source = Source(name='external_source', species=forcing_species)
    connection = Species2Species(source=source, sink=forcing_target,
                                rate='0 mol/yr', signal=signal, id='external_input')
    model.teaching_signal = signal
    model.teaching_connection = connection
    # Preserve the exact solver input for the supplied continuous budget audit.
    model.teaching_signal_time = model.time.copy()
    model.teaching_signal_flux = signal.m.copy()
    if forcing == 'OA':
        model.carbon_signal = signal
    else:
        model.alkalinity_signal = signal
    return model

fixed_cases = {label: build_complete_case(None if label == 'control' else label)
               for label in ('control', 'OA', 'OAE')}
assert fixed_cases['OA'].teaching_connection.sink is fixed_cases['OA'].CO2_At
assert fixed_cases['OAE'].teaching_connection.sink is fixed_cases['OAE'].L_b.TA

## B1. Inspect the forcing, then check the runs

The first plot shows the **actual input rates supplied to the solver**, before any response. Separate axes retain mol C versus TA-equivalent units; the third panel divides each rate by its own peak to compare timing only. The vertical line marks year 1800. Follow the rise, peak and decline and inspect whether input has ended when interpreting a later response. Area under a rate curve is an inventory; peak height is not the total input.

The supplied inventory check then precedes the three model integrations. Reuse 03's active boundary (atm plus dissolved ocn; sediment outside) and its weathering carbon flux $W_0$:

$$\frac{dC_{atm+ocn}}{dt}=I_C(t)+W_0-B_{net}(t),\qquad
\frac{dA_{ocn}}{dt}=I_A(t)+2W_0-2B_{net}(t).$$

Here $I_C$ is experimental carbon input [mol C/yr] and $I_A$ is experimental TA input [equivalents/yr]. $B_{net}$ is the signed net burial already defined in 03; negative values return sediment material to the active inventories. OA has $I_A=0$; pure-TA OAE has $I_C=0$.

Inspect the supplied budget errors once before interpreting results. They compare inventory changes with integrated boundary fluxes, allowing for numerical quadrature; passing does not require constant inventories or establish scientific realism.

In [ ]:
# Understand and run: inspect prescribed causes before interpreting responses.
from teaching_plots import plot_external_forcings
plot_external_forcings(fixed_cases, pulse_start=PULSE_START)
plt.show()

In [ ]:
# Understand and run: trace this supplied step and interpret its evidence.
from teaching_audits import integrate_forcing_history

forcing_rows = {}
for label, target in (('OA', oa_target_mol), ('OAE', oae_target_mol)):
    case = fixed_cases[label]
    time, flux = case.teaching_signal_time, case.teaching_signal_flux
    total = integrate_forcing_history(time, flux, time[-1])
    after_start = total - integrate_forcing_history(time, flux, PULSE_START)
    np.testing.assert_allclose(after_start, target, rtol=1e-3)
    forcing_rows[label] = {'post-1800 input (Pmol C or TA eq)': after_start / 1e15,
                           'whole-run input (Pmol C or TA eq)': total / 1e15}
display(pd.DataFrame(forcing_rows).T)

for label, case in fixed_cases.items():
    print('Running', label)
    run_model(case)
    postprocess_carbonate_horizons(case)
display(pd.DataFrame({label: audit_complete_model(case, label)
                      for label, case in fixed_cases.items()}).T)

## B2. Interpret matched responses

For each quantity use the <mark style="background-color: #fff0b3; color: #513d00; padding: 0.05em 0.2em; border-radius: 3px;"><strong>matched anomaly</strong></mark>

$$\Delta Y(t)=Y_{forced}(t)-Y_{control}(t).$$

Subtracting the control removes shared baseline drift. It isolates the response to the prescribed input within this model; it does not remove numerical error or validate omitted processes.

Read the atm CO2 and low-latitude surface pH anomalies first, then deep DIC and dissolution/net burial. Compare their signs and turning points with the forcing in B1: a response maximum need not coincide with the input-rate maximum. Surface chemistry, gas exchange, transport and sediment adjustment act together, with different response times.

**Compare mechanisms and timing, not raw amplitudes:** the two columns have different vertical scales, input species and total amounts. The overlaid normalized forcings establish common timing only; they do not make the interventions equally strong. Use B2 for C1–C2 and the depth/flux panels below for C3–C4.

In [ ]:
# Understand and run: trace this supplied step and interpret its evidence.
from teaching_plots import plot_matched_responses
plot_matched_responses(fixed_cases, pulse_start=PULSE_START)
plt.show()

## B3. Read critical depths and carbonate preservation

These complete figures reuse the same runs. Solid lines are model results; OA's dotted lines are archived digitizations of the published benchmark. Use them to assess reproduction, while B2's matched anomalies identify the forcing response.

| Panels | Read for |
| --- | --- |
| a / b / c | DIC, TA and pH: surface-to-deep chemical change |
| d / f | Air–sea transfer (positive into the ocn) and atm CO2 |
| e / h | Critical depths, dissolution and net burial: carbonate preservation |
| g | The forcing from B1, repeated here as a timing reference |

The three <mark style="background-color: #fff0b3; color: #513d00; padding: 0.05em 0.2em; border-radius: 3px;"><strong>critical depths</strong></mark> answer different questions. Calcite saturation $\Omega$ compares the calcium–carbonate ion product with its equilibrium solubility product; $\Omega<1$ means undersaturation.

| Depth | Physical meaning | What controls it here? |
| --- | --- | --- |
| $z_{sat}$: saturation horizon | Water has $\Omega_{calcite}=1$; deeper water is undersaturated | Current deep carbonate chemistry and pressure-dependent solubility |
| $z_{cc}$: compensation depth | Dissolution balances the arriving modern CaCO3 rain; none survives for burial | Current chemistry, dissolution kinetics and prescribed rain |
| $z_{snow}$: snowline | Deepest boundary of existing reactive carbonate sediment | Past deposition and dissolution; erosion takes time |

Thus undersaturated water does not imply immediate disappearance of all sediment carbonate. This distinction follows [Boudreau et al. (2010), *Carbonate compensation dynamics*](https://doi.org/10.1029/2009GL041847). Depth is positive downward in these definitions, but **panel e plots elevation $-z$**: upward means shallower. The lower limit of the implemented saturation horizon is 200 m; reaching that bound is not evidence that every surface water is undersaturated.

**Read e together with h.** When compensation depth and snowline separate, the sediment present need not be in equilibrium with current rain and chemistry. In this implementation, erosion depends on the finite existing reactive stock; during deepening the snowline is made to follow the compensation depth rapidly. It does not resolve gradual accumulation of a new sediment column. Treat that part of the OA/OAE contrast as a model assumption.

**Chemical carbonate compensation** means changing dissolution/preservation at fixed carbonate rain. It changes dissolved TA, unlike acid–base repartitioning among dissolved species. POC and PIC exports remain prescribed in all three cases: changing deep DIC alone cannot establish changing biological export.

The benchmark's transient depth separation and sediment response are discussed in [Boudreau et al. (2010), *Ongoing transients in carbonate compensation*, section 4.2](https://doi.org/10.1029/2009GB003654). Read the selected panels for the four answers below; no sediment-equation derivation or panel-by-panel report is required.

In [ ]:
# Understand and run: trace this supplied step and interpret its evidence.
from teaching_plots import plot_figure4
plot_figure4(fixed_cases['OA'], 'OA', reference=True,
             digitized=DIGITIZED, pulse_start=PULSE_START)
plt.show()

In [ ]:
# Understand and run: trace this supplied step and interpret its evidence.
plot_figure4(fixed_cases['OAE'], 'OAE', pulse_start=PULSE_START)
plt.show()

## C. Explain the scientific results in four short answers

Use two or three sentences per answer, referring to a curve and an approximate model year or interval. Read approximate values from the supplied plots; no extra calculations or runs are required.

<div style="background-color: #edf5ff; color: #173b61; border-left: 4px solid #3977b8; padding: 12px 16px; margin: 16px 0; border-radius: 4px;">

**Question — mechanisms, critical depths and OA/OAE asymmetry**

1. **Forcing and response:** revisit your sign prediction using B2. Compare the input-rate peak with the atm/surface response and one delayed deep response. Does declining input mean the perturbation is over?
2. **Carbon uptake and TA:** explain OAE's atm CO2 response without a direct carbon input. Contrast TA-conserving dissolved-species repartitioning with the changing ocn TA in OA; use the net-burial term in B1 and panel h as evidence.
3. **Critical depths and memory:** compare the directions and separation of $z_{sat}$, $z_{cc}$ and $z_{snow}$ in OA and OAE. What does the OA interval between compensation depth and snowline imply for old sediment, and why is OAE deepening not simply the reverse history? Relate the contrast to dissolution/net burial and the stated snowline assumption.
4. **Asymmetry and limits:** identify one difference beyond opposite response signs. Separate unequal forcing amounts/entry species from a chemical or sediment mechanism. Why can these runs establish neither an OAE amount that cancels OA nor stronger biological export from higher deep DIC? State what the late curves and benchmark overlay do (and do not) establish.

</div>

> **Your explanation:** replace this placeholder with your answer.

**You have completed 04.** Keep your forcing choices, checked budgets and four interpretations here.